# Suite DST — Distributions

A pattern is a predicate: `Distributions.Choices` weighs alternatives, its arms, and holds when the number of its arms
that hold satisfies its count; a distribution of values binds a symbol, within its body, to a value drawn from it, and
holds of data when the value its body equates the symbol with is in its support. `Generators.Sample` draws a store's
matches by what they weigh.

In [ ]:
from collections import Counter

from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Expressions.Framework import Terms
from mbse.Patterns import Constraints as C, Distributions as D, Generators as G, Predicates as P, register
from mbse.Schemas.Framework import JSON, Proxies, Schemas as S, Stores
from support import Contact, Phone, book, listed, raises

APerson = {"person": Contact}
person, age, x = E.variable("person"), E.variable("age"), E.variable("x")
HasName = (P.OfPredicate.Builder().name("HasName").symbols(APerson).parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())
Names = P.OfPredicate.Builder().name("Names").symbols(APerson).requires(D.Choices.Builder().arms(
    lambda a: a.weight(10).requires(HasName(person, "alice")),
    lambda a: a.weight(5).requires(HasName(person, "ben")),
    lambda a: a.weight(15).requires(HasName(person, "chermon")),
    lambda a: a.weight(7).requires(HasName(person, "davi")),
).decreasing().create()).create()

## DST-01 · Choices are built fluently: arms, a count and a precedence

In [ ]:
choices = Names.rule
assert isinstance(choices, D.Choices) and choices.decreasing is True and [a.weight for a in choices.arms] == [10.0, 5.0, 15.0, 7.0]
assert all(arm.condition.predicate is HasName for arm in choices.arms)  # one predicate, by reference
assert (choices.count.name, choices.count.binds(), D.Count().binds()) == ("c", ("c",), ())
assert Terms.same(choices.count.condition, E.variable("c").ge(1).data)  # at least one arm, by default
both = D.Choices.Builder().arms(lambda a: a.weight(1).requires(person.has("name"), person.has("age"))).count(
    lambda n: n == 1).create()  # an arm's conditions are conjoined; the count is read from the function
assert Terms.same(both.arms[0].condition, person.has("name").and_(person.has("age")).data) and both.decreasing is None
assert both.count.name == "n" and Terms.same(both.count.condition, E.variable("n").eq(1).data)
grown = D.Choices.Builder(both).arms(lambda a: a.weight(2).requires(True)).clone()
assert len(grown.arms) == 2 and len(both.arms) == 1 and grown.count is both.count  # the count is kept
assert D.Choices.Builder(grown).decreasing().update() is grown and grown.decreasing is True
arm = D.Arm.resolve(lambda a: a.weight(2).requires(True).requires(False))
assert arm.weight == 2.0 and Terms.same(arm.condition, E.literal(True).and_(False).data) and D.Arm.resolve(arm) is arm
assert Terms.same(D.Arm.Builder(arm).requires(x).clone().condition, E.operation("and", arm.condition, x).data)
assert D.Arm.Builder(arm).weight(3).update() is arm and arm.weight == 3.0
assert P.DIALECT.validate(D.Arm.Builder().weight(1).create()) == ["an arm needs a condition"]
with raises(TypeError, match="expected an arm or a callable taking its builder, got 'x'"):
    D.Arm.resolve("x")
with raises(TypeError, match="a choices' count is a function of one number"):
    D.Choices.Builder().count(lambda a, b: True)
assert P.DIALECT.validate(Names, core=True) == []
assert P.DIALECT.validate(D.Choices.Builder().create()) == ["a choices needs an arm"]
assert P.DIALECT.validate(D.Choices(None, both.count, (E.literal(1).data,))) == ["a choices' arguments after its count are arms"]
assert P.DIALECT.validate(D.Choices(None, E.literal(1).data, both.arms), bound={"person"}) == ["a choices' count must be a count"]
assert P.DIALECT.validate(D.Choices.Builder().arms(lambda a: a.weight(-1).requires(True)).create()) == [
    "argument 0: an arm's weight must be positive, got -1.0"]

## DST-02 · Choices hold when their count does; a match weighs what its arms say

In [ ]:
store = book()
alice, ben, zed = (store.Contact().name(n).create() for n in ("alice", "ben", "zed"))
evaluate = P.Evaluator(store)
assert [P.holds(evaluate, Names.rule, {"person": p}) for p in (alice, ben, zed)] == [True, True, False]
Two = D.Choices.Builder().arms(lambda a: a.weight(1).requires(person.has("name")),
                               lambda a: a.weight(2).requires(HasName(person, "alice"))).count(lambda c: c == 2).create()
assert [evaluate(Two, {"person": p}) for p in (alice, ben)] == [True, False]  # both arms hold of alice only
unknown = D.Choices.Builder().arms(lambda a: a.weight(1).requires(x), lambda a: a.weight(1).requires(True)).create()
assert evaluate(unknown, {"x": None}) is True  # one arm holds: at least one, whatever the other
assert evaluate(D.Choices.Builder(unknown).count(lambda c: c == 1).clone(), {"x": None}) is None  # 1 or 2
with raises(TypeError, match="a predicate must be a bool, got int"):
    evaluate(D.Choices.Builder().arms(lambda a: a.weight(1).requires(3)).create())
assert [evaluate.weigh(Names.rule, {"person": p}) for p in (alice, ben, zed)] == [10.0, 5.0, 0.0]
def overlapping():
    return D.Choices.Builder().arms(lambda a: a.weight(3).requires(person.has("name")),
                                    lambda a: a.weight(2).requires(HasName(person, "alice")))


assert evaluate.weigh(overlapping().decreasing().create(), {"person": alice}) == 3.0  # the first that holds
assert evaluate.weigh(overlapping().create(), {"person": alice}) == 5.0  # without precedence, every arm that holds
none = D.Choices.Builder().arms(lambda a: a.weight(1).requires(person.has("pager"))).count(lambda c: c == 0).decreasing().create()
assert evaluate(none, {"person": alice}) is True and evaluate.weigh(none, {"person": alice}) == 0.0  # no arm to fall under
Nested = P.OfPredicate.Builder().symbols(APerson).requires(person.has("name")).requires(Names.rule).requires(
    D.Choices.Builder().arms(lambda a: a.weight(2).requires(D.Choices.Builder().arms(
        lambda b: b.weight(4).requires(True)).create())).create()).create()
assert evaluate.weigh(Nested.rule, {"person": alice}) == 10.0 * 2.0 * 4.0  # conjuncts multiply, arms nest
applied = P.OfPredicate.Builder().symbols({"who": Contact}).requires(Names(E.variable("who"))).create()
assert evaluate.weigh(applied.rule, {"who": ben}) == 5.0  # through an application

## DST-03 · Distributions of values bind a symbol, and are built fluently

In [ ]:
normal = D.Normal.Builder().symbol("age").mean(70).deviation(8).rounded().requires(
    lambda person, age: person.age == age).create()
assert (normal.symbol, normal.rounded, normal.mean.value, normal.deviation.value) == ("age", True, 70, 8)
assert normal.binds() == ("age",) and Terms.same(normal.body, person.age.eq(age).data)  # read from the function
assert D.Normal().binds() == ()  # without a symbol, nothing
builder = D.Uniform.Builder().symbol("n").low(1).high(6)
assert Terms.same(builder.n.data, E.variable("n").data)  # the builder gives its symbol as a variable
with raises(AttributeError, match="m"):
    builder.m
uniform = builder.requires(person.age.eq(builder.n), person.has("name")).create()  # specs, conjoined
assert Terms.same(uniform.body, person.age.eq(E.variable("n")).and_(person.has("name")).data)
assert Terms.same(D.Uniform.Builder(uniform).requires(True).clone().body, E.operation("and", uniform.body, True).data)
assert D.Uniform.Builder(uniform).high(9).update() is uniform and uniform.high.value == 9
categorical = D.Categorical.Builder().symbol("name").option(3, "ann").option(1, "bo").requires(
    person.name.eq(E.variable("name"))).create()
assert [(o.weight, o.value.value) for o in categorical.options] == [(3.0, "ann"), (1.0, "bo")]
poisson = D.Poisson.Builder().symbol("k").rate(1.5).create()
geometric = D.Geometric.Builder().symbol("k").probability(0.5).create()
assert [D.domain(d) for d in (
    normal, D.Normal.Builder().mean(1.0).deviation(1.0).create(), poisson, geometric, uniform,
    D.Uniform.Builder().low(1).high(2.5).create(), D.Uniform.Builder().low(x).high(2).create(), categorical,
    D.Categorical.Builder().option(1, "a").option(2, 3).create(), E.literal(True).data, x.data)] == [
    "int", "float", "int", "int", "int", "float", None, "str", None, "bool", None]
assert P.DIALECT.validate(normal, bound={"person"}) == []
assert P.DIALECT.validate(D.Normal.Builder().mean(1).deviation(1).create()) == [
    "a normal needs a symbol", "a normal needs a body"]
assert P.DIALECT.validate(D.Categorical.Builder().symbol("v").requires(True).create()) == ["a categorical needs an option"]
assert P.DIALECT.validate(D.Categorical("v", E.literal(True).data, (E.literal(1).data,))) == [
    "a categorical's arguments after its body are options"]
assert P.DIALECT.validate(D.Categorical.Builder().symbol("v").option(-1, 1).requires(True).create()) == [
    "argument 0: an option's weight must be positive, got -1.0"]
assert Terms.same(D.witness(normal), person.age.data)  # what the body equates the symbol with
looping = D.Poisson.Builder().symbol("k").requires(E.variable("k").eq(E.variable("k").add(1))).create()
assert D.witness(looping) is None  # a witness does not read the symbol itself

## DST-04 · A distribution holds of data when its witness is in its support

In [ ]:
store = book()
ages = (30, -2, None)
people = [store.Contact().age(a).create() if a is not None else store.Contact().create() for a in ages]
evaluate = P.Evaluator(store)


def held(make):
    rule = make().symbol("v").requires(person.age.eq(E.variable("v"))).create()
    return [P.holds(evaluate, rule, {"person": p}) for p in people]


assert held(lambda: D.Normal.Builder().mean(0).deviation(1).rounded()) == [True, True, None]  # any int; no witness
assert held(lambda: D.Normal.Builder().mean(0).deviation(1)) == [True, True, None]  # any number: 30 is 30.0
assert held(lambda: D.Uniform.Builder().low(18).high(64)) == [True, False, None]
assert held(lambda: D.Poisson.Builder().rate(2)) == [True, False, None] == held(lambda: D.Geometric.Builder().probability(0.5))
assert held(lambda: D.Categorical.Builder().option(1, 30).option(1, x)) == [True, False, None]
floats = D.Uniform.Builder().symbol("v").low(0.0).high(1.0).requires(E.variable("v").eq(x)).create()
assert [evaluate(floats, {"x": v}) for v in (0.0, 0.5, 1.0, "a")] == [True, True, False, False]  # [low, high)
extra = D.Uniform.Builder().symbol("v").low(0).high(100).requires(person.age.eq(E.variable("v")), E.variable("v").gt(50)).create()
assert evaluate(extra, {"person": people[0]}) is False  # in the support, but the body does not hold
assert evaluate(D.Normal.Builder().symbol("v").mean(0).deviation(1).requires(True).create()) is None  # no witness
Wrong = P.OfPredicate.Builder().name("Wrong").symbols(APerson).requires(
    D.Normal.Builder().symbol("v").mean(0).deviation(1).requires(person.age.eq(E.variable("v"))).create(),
    ).requires(D.Categorical.Builder().symbol("w").option(1, 2).requires(E.variable("w").eq(person.name)).create(),
    ).requires(D.Normal.Builder().symbol("u").mean(0).deviation(1).requires(x.age.eq(E.variable("u")), person.age.eq(1)).create()
    ).create()
assert D.typing(Wrong) == ["person.age is int, but its distribution gives float", "person.name is str, but its distribution gives int"]
with raises(ValueError, match="predicate 'Wrong': person.age is int, but its distribution gives float"):
    C.check([Wrong])
Tagged = S.OfObject.Builder().name("Tagged").ref().properties(
    lambda p: p.name("tags").of(lambda t: t.as_indexed(lambda i: i.of(lambda t: t.as_native(str))))).create()
Tags = P.OfPredicate.Builder().symbols({"it": Tagged}).requires(D.Poisson.Builder().symbol("v").rate(1).requires(
    E.variable("it").tags.eq(E.variable("v"))).create()).create()
assert D.typing(Tags) == []  # a property that is not a native: its type is not checked
Fine = P.OfPredicate.Builder().symbols(APerson).requires(D.Uniform.Builder().symbol("v").low(1).high(6).requires(
    person.age.eq(E.variable("v")), E.variable("v").eq(3)).create()).create()
assert D.typing(Fine) == []  # an int, for an int; an equality that sets no property is passed over
outer = P.OfPredicate.Builder().symbols(APerson).requires(Wrong(person)).create()
assert D.typing(outer) == []  # an applied predicate's distributions are its own to check
assert D.typing(P.OfPredicate.Builder().create()) == []  # no rule, nothing to check

## DST-05 · Sample draws the store's matches in proportion to what they weigh

In [ ]:
store = book()
listed(store, *(store.Contact().name(n).create() for n in ("alice", "ben", "chermon", "davi", "zed")))
drawn = G.Sample(store, Names, Stores.PCG32(42))
names = Counter(next(drawn)["person"].name for _ in range(3700))
assert set(names) == {"alice", "ben", "chermon", "davi"}  # zed weighs nothing
assert all(abs(names[n] - 100 * w) < 100 for n, w in (("alice", 10), ("ben", 5), ("chermon", 15), ("davi", 7))), names
again = [next(G.Sample(store, Names, Stores.PCG32(42)))["person"].name for _ in range(3)]
assert again == [again[0]] * 3  # the same seed, the same draws
first = [m["person"].name for m, _ in zip(G.Sample(store, Names, Stores.PCG32(42)), range(6))]
assert first != [m["person"].name for m, _ in zip(G.Sample(store, Names, Stores.PCG32(42).split("other")), range(6))]
nobody = book()
listed(nobody, nobody.Contact().name("zed").create())
with raises(ValueError, match="no match of the store's data has a weight"):
    next(G.Sample(nobody, Names, Stores.PCG32(1)))
with raises(ValueError, match="the predicate cannot be drawn from: a predicate with parameters is drawn from where it is applied"):
    G.Sample(store, HasName, Stores.PCG32(1))

## DST-06 · Patterns are data: a predicate used by several arms is written once

In [ ]:
store = C.OfStore(book())
text = JSON.ToJSON(store).Reachable(P.OfPredicate.Schema, Names)
assert text.count('"name": "HasName"') == 1 and '"$schema": "Patterns.OfChoices"' in text and '"decreasing": true' in text
copy = JSON.FromJSON(store).Reachable(P.OfPredicate.Schema, text)
assert Terms.same(copy, Names) and len({id(arm.condition.predicate) for arm in copy.rule.arms}) == 1
People = P.OfPredicate.Builder().symbols(APerson).requires(D.Choices.Builder().arms(lambda a: a.weight(1).requires(
    D.Normal.Builder().symbol("age").mean(70).deviation(8).rounded().requires(lambda person, age: person.age == age).create(),
    D.Categorical.Builder().symbol("name").option(3, "ann").option(1, "bo").requires(person.name.eq(E.variable("name"))).create(),
)).create()).create()
text = JSON.ToJSON(store).Reachable(P.OfPredicate.Schema, People)
assert '"$schema": "Patterns.OfNormal"' in text and '"rounded": true' in text and '"$schema": "Patterns.OfOption"' in text
assert Terms.same(JSON.FromJSON(store).Reachable(P.OfPredicate.Schema, text), People)

## DST-07 · Patterns meet quantifiers, sets and stores

In [ ]:
store = book()
listed(store, *(store.Contact().name(n).create() for n in ("ann", "bob")))
ann = store.Contact().name("ann").phones(lambda e: e.phone(lambda q: q.number("1"))).create()
listed(store, ann)
p = E.variable("p")
owns = P.Exists(lambda q: q.symbols({"p": Phone}).requires(P.Contains(person.phones, lambda e: e.phone == p)))
Owning = P.OfPredicate.Builder().name("Owning").symbols(APerson).requires(D.Choices.Builder().arms(
    lambda a: a.weight(2).requires(owns), lambda a: a.weight(1).requires(person.has("name"))).decreasing().create()).create()
evaluate = P.Evaluator(store)
assert [evaluate.weigh(Owning.rule, {"person": who}) for who in store.extent("Contact")] == [1.0, 1.0, 2.0]  # a link weighs
predicates = C.check([Names, Owning])  # patterns in a set, checked as predicates are
assert [q.name for q in predicates.predicates] == ["Names", "Owning"]
schemas = C.OfStore(book())
copy = JSON.FromJSON(schemas).Reachable(C.OfSet.Schema, JSON.ToJSON(schemas).Reachable(C.OfSet.Schema, predicates))
assert all(Terms.same(a, b) for a, b in zip(copy.predicates, predicates.predicates))
proxies = register(Proxies.OfStore())
assert {"Patterns.OfChoices", "Patterns.OfArm", "Patterns.OfCount", "Patterns.OfNormal", "Patterns.OfOption"} <= set(proxies.names())
Not = P.OfPredicate.Builder().symbols(APerson).forbids(Names.rule).create()  # a pattern's negation is a predicate
assert [P.holds(evaluate, Not.rule, {"person": who}) for who in store.extent("Contact")] == [True, True, True]
assert evaluate.weigh(Not.rule, {"person": ann}) == 1.0  # choices under not weigh nothing